Import libraries and load datasets

In [17]:
import pandas as pd

# Try reading using the 'python' engine and skipping bad lines if necessary
talks = pd.read_csv('/content/cleaned_ted_talks.csv', engine='python', on_bad_lines='skip')

print(talks.shape)
talks.head()

(894, 18)


,comments,description,duration,event,film_date,languages,main_speaker,name,num_speaker,published_date,ratings,related_talks,speaker_occupation,tags,title,url,views,transcript
0,4553,Sir Ken Robinson makes an entertaining and pro...,1164,TED2006,1140825600,60,Ken Robinson,Ken Robinson: Do schools kill creativity?,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 19645}, {...","[{'id': 865, 'hero': 'https://pe.tedcdn.com/im...",Author/educator,"['children', 'creativity', 'culture', 'dance',...",Do schools kill creativity?,https://www.ted.com/talks/ken_robinson_says_sc...,47227110,Good morning. How are you?(Laughter)It's been ...
1,265,With the same humor and humanity he exuded in ...,977,TED2006,1140825600,43,Al Gore,Al Gore: Averting the climate crisis,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 544}, {'i...","[{'id': 243, 'hero': 'https://pe.tedcdn.com/im...",Climate advocate,"['alternative energy', 'cars', 'climate change...",Averting the climate crisis,https://www.ted.com/talks/al_gore_on_averting_...,3200520,"Thank you so much, Chris. And it's truly a gre..."
2,124,New York Times columnist David Pogue takes aim...,1286,TED2006,1140739200,26,David Pogue,David Pogue: Simplicity sells,1,1151367060,"[{'id': 7, 'name': 'Funny', 'count': 964}, {'i...","[{'id': 1725, 'hero': 'https://pe.tedcdn.com/i...",Technology columnist,"['computers', 'entertainment', 'interface desi...",Simplicity sells,https://www.ted.com/talks/david_pogue_says_sim...,1636292,"(Music: ""The Sound of Silence,"" Simon & Garfun..."
3,200,"In an emotionally charged talk, MacArthur-winn...",1116,TED2006,1140912000,35,Majora Carter,Majora Carter: Greening the ghetto,1,1151367060,"[{'id': 3, 'name': 'Courageous', 'count': 760}...","[{'id': 1041, 'hero': 'https://pe.tedcdn.com/i...",Activist for environmental justice,"['MacArthur grant', 'activism', 'business', 'c...",Greening the ghetto,https://www.ted.com/talks/majora_carter_s_tale...,1697550,If you're here today — and I'm very happy that...
4,593,You've never seen data presented like this. Wi...,1190,TED2006,1140566400,48,Hans Rosling,Hans Rosling: The best stats you've ever seen,1,1151440680,"[{'id': 9, 'name': 'Ingenious', 'count': 3202}...","[{'id': 2056, 'hero': 'https://pe.tedcdn.com/i...",Global health expert; data visionary,"['Africa', 'Asia', 'Google', 'demo', 'economic...",The best stats you've ever seen,https://www.ted.com/talks/hans_rosling_shows_t...,12005869,"About 10 years ago, I took on the task to teac..."


In [18]:
interactions = pd.read_csv('/content/07_user_interaction_data.csv')

interactions.shape
interactions.head()

,user_id,ted_talk_id,talk_title,rating,liked,watched,saved
0,U001,TED_SAMPLE_001,Do schools kill creativity?,5.0,True,True,True
1,U001,TED_SAMPLE_002,Averting the climate crisis,4.0,True,True,True
2,U001,TED_SAMPLE_003,Simplicity sells,5.0,True,True,True
3,U001,TED_SAMPLE_004,Greening the ghetto,2.0,False,True,False
4,U001,TED_SAMPLE_005,The best stats you've ever seen,1.0,False,True,False


Prepare TED Talk content

We combine title, tags, description, and transcript to create content for recommendations.

In [19]:
# Find available text columns
text_columns = []

for col in ["title", "tags", "description", "transcript"]:
    if col in talks.columns:
        text_columns.append(col)

# Find title column
if "title" in talks.columns:
    title_col = "title"
elif "talk_title" in talks.columns:
    title_col = "talk_title"
else:
    raise ValueError("TED Talk title column not found.")

# Find ID column
if "ted_talk_id" in talks.columns:
    id_col = "ted_talk_id"
elif "talk_id" in talks.columns:
    id_col = "talk_id"
elif "id" in talks.columns:
    id_col = "id"
else:
    talks["ted_talk_id"] = talks.index.astype(str)
    id_col = "ted_talk_id"

# Prepare IDs and text
talks["talk_key"] = talks[id_col].astype(str).str.strip()

talks["content"] = talks[text_columns].fillna("").astype(str).agg(" ".join, axis=1)

# Remove empty content
talks = talks[talks["content"].str.strip() != ""].copy()

# Create TF-IDF model
tfidf = TfidfVectorizer(stop_words="english", max_features=5000)

content_matrix = tfidf.fit_transform(talks["content"])

print("Content-based model created!")
print("Total talks:", len(talks))

Content-based model created!
Total talks: 894


Prepare user preferences and collaborative model

This cell connects interactions to the cleaned TED Talks dataset. It uses talk titles as a fallback when Task 7 IDs do not match the cleaned dataset IDs.

In [21]:
from sklearn.metrics.pairwise import cosine_similarity

# Match interaction IDs to TED Talk IDs
id_map = dict(zip(talks["talk_key"], talks["talk_key"]))

# Create title matching dictionary
title_map = {}

for _, row in talks.iterrows():
    title_map[str(row[title_col]).strip().lower()] = row["talk_key"]

# Find interaction title column
if "talk_title" in interactions.columns:
    interaction_title = "talk_title"
else:
    interaction_title = None

# Match each interaction to a talk
interactions["talk_key"] = interactions["ted_talk_id"].astype(str)

for i, row in interactions.iterrows():

    if row["talk_key"] in id_map:
        continue

    if interaction_title:
        title = str(row[interaction_title]).strip().lower()

        if title in title_map:
            interactions.loc[i, "talk_key"] = title_map[title]
        else:
            interactions.loc[i, "talk_key"] = None

# Keep only matched talks
interactions = interactions.dropna(subset=["talk_key"])

# Create user-item rating matrix
interactions["rating"] = pd.to_numeric(
    interactions["rating"], errors="coerce"
)

rating_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_key",
    values="rating"
)

# Calculate user similarity
similarity = cosine_similarity(rating_matrix.fillna(0))

similarity_matrix = pd.DataFrame(
    similarity,
    index=rating_matrix.index,
    columns=rating_matrix.index
)

print("Collaborative model created!")
display(rating_matrix)
display(similarity_matrix)

Collaborative model created!


talk_key,0,1,2,3,4,5
user_id,,,,,,
U001,5.0,4.0,5.0,2.0,1.0,NaN
U002,5.0,4.0,4.0,2.0,1.0,5.0
U003,1.0,2.0,1.0,5.0,5.0,1.0
U004,4.0,5.0,4.0,1.0,2.0,NaN
U005,2.0,1.0,2.0,5.0,4.0,1.0


user_id,U001,U002,U003,U004,U005
user_id,,,,,
U001,1.000000,0.839760,0.518737,0.964617,0.631494
U002,0.839760,1.000000,0.525418,0.816951,0.615516
U003,0.518737,0.525418,1.000000,0.555112,0.964452
U004,0.964617,0.816951,0.555112,1.000000,0.604642
U005,0.631494,0.615516,0.964452,0.604642,1.000000


Content-based recommendation function

This finds talks similar to the talks a user has liked or rated highly.

In [29]:
def content_recommend(user_id, top_n=5):
    user_data = interactions[interactions["user_id"] == user_id]

    if user_data.empty:
        return pd.DataFrame()

    # Find talks liked or rated 4 and above
    liked = user_data[
        (user_data["rating"] >= 4) |
        (user_data["liked"].astype(str).str.lower() == "true")
    ]

    if liked.empty:
        return pd.DataFrame()

    # Find liked talk positions
    liked_positions = talks[
        talks["talk_key"].isin(liked["talk_key"])
    ].index.tolist()

    if not liked_positions:
        return pd.DataFrame()

    # Create user content profile and convert the matrix mean to a flat 1D array
    user_profile = np.asarray(content_matrix[liked_positions].mean(axis=0)).reshape(1, -1)

    # Calculate similarity with all talks
    scores = cosine_similarity(
        user_profile,
        content_matrix
    ).flatten()

    result = talks[["talk_key", title_col]].copy()
    result["content_score"] = scores

    # Exclude already rated talks
    rated = user_data["talk_key"].tolist()
    result = result[~result["talk_key"].isin(rated)]

    return result.sort_values(
        "content_score", ascending=False
    ).head(top_n)

Collaborative recommendation function

This uses similar users to predict ratings for talks the selected user has not rated.

In [30]:
def collaborative_recommend(user_id, top_n=5):
    if user_id not in rating_matrix.index:
        return pd.DataFrame()

    user_ratings = rating_matrix.loc[user_id]
    similar_users = similarity_matrix.loc[user_id].drop(user_id)

    results = []

    # Iterate through all talks in the main dataset instead of only rating_matrix columns
    for talk in talks["talk_key"]:
        # Skip if the user has already rated this talk in our interaction data
        if talk in user_ratings.index and pd.notna(user_ratings[talk]):
            continue

        total = 0
        weight = 0

        # If the talk exists in the rating matrix, we can predict a score using neighbor ratings
        if talk in rating_matrix.columns:
            for other_user in similar_users.index:
                score = similar_users[other_user]
                rating = rating_matrix.loc[other_user, talk]

                if score > 0 and pd.notna(rating):
                    total += score * rating
                    weight += score

        if weight > 0:
            predicted = total / weight
            results.append([talk, predicted])

    result = pd.DataFrame(
        results,
        columns=["talk_key", "predicted_rating"]
    )

    return result.sort_values(
        "predicted_rating", ascending=False
    ).head(top_n)

Hybrid recommendation function

This is the main function. It combines both recommendation models, removes duplicates, and ranks the results.

In [34]:
def hybrid_recommend(user_id, top_n=5):

    # Get recommendations from both models
    content = content_recommend(user_id, top_n=20)
    collaborative = collaborative_recommend(user_id, top_n=20)

    if content.empty and collaborative.empty:
        return pd.DataFrame()

    # Initialize empty dataframes with correct columns if one of the recommendation sets is empty
    if content.empty:
        content = pd.DataFrame(columns=["talk_key", "content_score"])
    else:
        content = content[["talk_key", "content_score"]]

    if collaborative.empty:
        collaborative = pd.DataFrame(columns=["talk_key", "collab_score"])
    else:
        # Normalize collaborative rating to 0–1
        collaborative["collab_score"] = (
            collaborative["predicted_rating"] / 5
        )
        collaborative = collaborative[["talk_key", "collab_score"]]

    # Combine both results
    result = pd.merge(
        content,
        collaborative,
        on="talk_key",
        how="outer"
    )

    # Fill missing scores with 0
    result = result.fillna(0)

    # Calculate hybrid score
    result["hybrid_score"] = (
        0.5 * result["content_score"] +
        0.5 * result["collab_score"]
    )

    # Remove duplicate talks
    result = result.drop_duplicates(subset="talk_key")

    # Exclude already rated talks
    rated = rating_matrix.loc[user_id].dropna().index.tolist()
    result = result[~result["talk_key"].isin(rated)]

    # Add TED Talk titles
    result = result.merge(
        talks[["talk_key", title_col]],
        on="talk_key",
        how="left"
    )

    return result.sort_values(
        "hybrid_score", ascending=False
    ).head(top_n)

Test different users and compare results

In [35]:
# Test three users to verify they get valid recommendation tables
test_users = rating_matrix.index[:3]

for user in test_users:
    print(f"\n=== RECOMMENDATIONS FOR {user} ===")

    print("\n1. Content-Based Recommendations:")
    display(content_recommend(user, top_n=5))

    print("\n2. Collaborative Recommendations:")
    display(collaborative_recommend(user, top_n=5))

    print("\n3. Hybrid Recommendations:")
    display(hybrid_recommend(user, top_n=5))


=== RECOMMENDATIONS FOR U001 ===

1. Content-Based Recommendations:


,talk_key,title,content_score
21,21,Nerdcore comedy,0.494534
568,568,A one-man world summit,0.464847
692,692,Bring on the learning revolution!,0.462421
370,370,Design and discovery,0.451412
97,97,A comic sendup of TED2006,0.435542



2. Collaborative Recommendations:


,talk_key,predicted_rating
0,5,2.687967



3. Hybrid Recommendations:


,talk_key,content_score,collab_score,hybrid_score,title
11,5,0.367443,0.537593,0.452518,Why we do what we do
2,21,0.494534,0.000000,0.247267,Nerdcore comedy
12,568,0.464847,0.000000,0.232424,A one-man world summit
17,692,0.462421,0.000000,0.231211,Bring on the learning revolution!
6,370,0.451412,0.000000,0.225706,Design and discovery


... (10 outputs left out)

,talk_key,predicted_rating



3. Hybrid Recommendations:


/tmp/ipykernel_727/794113909.py:34: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  result = result.fillna(0)


,talk_key,content_score,collab_score,hybrid_score,title
10,502,0.516496,0,0.258248,Let my dataset change your mindset
2,117,0.394289,0,0.197145,New insights on poverty
18,784,0.338244,0,0.169122,The good news of the decade? We're winning the...
15,726,0.325620,0,0.162810,"Global population growth, box by box"
14,63,0.316660,0,0.158330,My wish: Rebuilding Rwanda


Validate the hybrid model

In [26]:
user = rating_matrix.index[0]

result = hybrid_recommend(user, top_n=5)

# Check duplicate recommendations
print("Duplicate talks:", result["talk_key"].duplicated().sum())

# Check descending order
print(
    "Correct ranking:",
    result["hybrid_score"].is_monotonic_decreasing
)

# Check already-rated talks
rated = rating_matrix.loc[user].dropna().index

print(
    "Rated talks included:",
    result["talk_key"].isin(rated).any()
)

display(result)

Duplicate talks: 0
Correct ranking: True
Rated talks included: False


,talk_key,content_score,collab_score,hybrid_score,title
11,5,0.367443,0.537593,0.452518,Why we do what we do
2,21,0.494534,0.000000,0.247267,Nerdcore comedy
12,568,0.464847,0.000000,0.232424,A one-man world summit
17,692,0.462421,0.000000,0.231211,Bring on the learning revolution!
6,370,0.451412,0.000000,0.225706,Design and discovery
